# Stable Diffusion XL — ใช้งานง่ายใน Google Colab 🎨

สร้างภาพด้วย Gradio UI ในหน้าเดียว รองรับ Animagine XL 4.0, SDXL Base 1.0 และ SSD-1B

## วิธีเริ่มใช้งาน
1. เปิดโน้ตบุ๊กใน Google Colab แล้วเลือก **Runtime → Change runtime type → GPU**
2. รันเซลล์ติดตั้ง dependencies หนึ่งครั้ง หาก Colab ขอให้รีสตาร์ต ให้รีสตาร์ตแล้วรันเซลล์ถัดไป
3. รันเซลล์ตั้งค่า และเซลล์เปิดหน้าจอใช้งาน
4. เลือกโมเดลและ Scheduler แล้วกด **Load model** ก่อนกรอก prompt และกด **Generate**

> ลิงก์ Gradio ที่แชร์จาก Colab เป็นลิงก์สาธารณะชั่วคราว อย่าใส่ข้อมูลส่วนตัวใน prompt
> การสร้างภาพใช้ VRAM มาก หากหน่วยความจำไม่พอ ให้ลด Batch size หรือขนาดภาพ

In [ ]:
# ติดตั้งเฉพาะไลบรารีที่จำเป็น โดยใช้ PyTorch ที่ติดมากับ Colab
!pip -q install -U diffusers transformers accelerate safetensors gradio

In [ ]:
import gc
import random

import torch
import gradio as gr
from diffusers import (
    StableDiffusionXLPipeline,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    UniPCMultistepScheduler,
    DDIMScheduler,
)

if not torch.cuda.is_available():
    raise RuntimeError("ไม่พบ GPU กรุณาเลือก Runtime → Change runtime type → GPU แล้วรันเซลล์ใหม่")

MODELS = {
    "Animagine XL 4.0 (แนะนำ)": "cagliostrolab/animagine-xl-4.0",
    "Stable Diffusion XL Base 1.0": "stabilityai/stable-diffusion-xl-base-1.0",
    "SSD-1B (ทำงานเร็ว)": "segmind/SSD-1B",
}
SCHEDULERS = {
    "Euler Ancestral": EulerAncestralDiscreteScheduler,
    "Euler": EulerDiscreteScheduler,
    "UniPC (แนะนำ)": UniPCMultistepScheduler,
    "DDIM": DDIMScheduler,
}
pipe = None

def load_model(model_label, scheduler_label):
    global pipe
    if not torch.cuda.is_available():
        raise gr.Error("ไม่พบ GPU กรุณาเปลี่ยน runtime เป็น GPU")

    # เคลียร์โมเดลเดิมก่อนโหลด เพื่อลดปัญหา VRAM เต็ม
    pipe = None
    gc.collect()
    torch.cuda.empty_cache()

    try:
        model_id = MODELS[model_label]
        scheduler = SCHEDULERS[scheduler_label].from_pretrained(
            model_id, subfolder="scheduler"
        )
        new_pipe = StableDiffusionXLPipeline.from_pretrained(
            model_id, scheduler=scheduler, torch_dtype=torch.float16
        ).to("cuda")
        new_pipe.enable_vae_slicing()
        pipe = new_pipe
        return f"พร้อมใช้งาน: {model_label} / {scheduler_label}"
    except Exception as exc:
        pipe = None
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(f"โหลดโมเดลไม่สำเร็จ: {exc}")

def generate_images(prompt, negative_prompt, width, height, steps, cfg_scale, batch_size, seed):
    if pipe is None:
        raise gr.Error("กรุณาเลือกโมเดลและกด Load model ก่อน")
    if not prompt or not prompt.strip():
        raise gr.Error("กรุณากรอก Prompt")

    seed = int(seed)
    if seed == -1:
        seed = random.randint(0, 2**31 - 1)
    elif seed < 0:
        raise gr.Error("Seed ต้องเป็น -1 (สุ่ม) หรือเลขตั้งแต่ 0 ขึ้นไป")

    try:
        generator = torch.Generator(device="cuda").manual_seed(seed)
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            width=int(width),
            height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg_scale),
            num_images_per_prompt=int(batch_size),
            generator=generator,
        )
        return result.images, f"Seed: {seed}"
    except Exception as exc:
        raise gr.Error(f"สร้างภาพไม่สำเร็จ: {exc}")


In [ ]:
sizes = [768, 832, 1024, 1216]

with gr.Blocks(theme=gr.themes.Soft(), title="SDXL Studio") as demo:
    gr.Markdown("# Stable Diffusion XL Studio\nเลือกโมเดลแล้วโหลดก่อนเริ่มสร้างภาพ")

    with gr.Accordion("1. ตั้งค่าโมเดล", open=True):
        with gr.Row():
            model_choice = gr.Dropdown(choices=list(MODELS), value="Animagine XL 4.0 (แนะนำ)", label="โมเดล")
            scheduler_choice = gr.Dropdown(choices=list(SCHEDULERS), value="UniPC (แนะนำ)", label="Scheduler")
        load_button = gr.Button("Load model", variant="secondary")
        status = gr.Textbox(label="สถานะ", value="ยังไม่ได้โหลดโมเดล", interactive=False)
        load_button.click(load_model, inputs=[model_choice, scheduler_choice], outputs=status)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 2. กำหนดภาพ")
            prompt = gr.Textbox(label="Prompt", lines=4, placeholder="อธิบายภาพที่ต้องการ")
            negative_prompt = gr.Textbox(label="Negative prompt", lines=2, placeholder="สิ่งที่ไม่ต้องการในภาพ (ไม่บังคับ)")
            with gr.Row():
                width = gr.Dropdown(choices=sizes, value=832, label="กว้าง")
                height = gr.Dropdown(choices=sizes, value=1216, label="สูง")
            with gr.Row():
                steps = gr.Slider(minimum=15, maximum=50, step=1, value=28, label="จำนวน Steps")
                cfg_scale = gr.Slider(minimum=1, maximum=12, step=0.5, value=5, label="CFG Scale")
            with gr.Row():
                batch_size = gr.Slider(minimum=1, maximum=4, step=1, value=1, label="จำนวนภาพ (Batch)")
                seed = gr.Number(value=-1, precision=0, label="Seed (-1 = สุ่ม)")
            generate_button = gr.Button("Generate", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### 3. ผลลัพธ์")
            gallery = gr.Gallery(label="ภาพที่สร้าง", columns=2, height="auto")
            seed_info = gr.Textbox(label="ข้อมูล Seed", interactive=False)

    generate_button.click(
        fn=generate_images,
        inputs=[prompt, negative_prompt, width, height, steps, cfg_scale, batch_size, seed],
        outputs=[gallery, seed_info],
    )

demo.queue().launch(share=True, inline=True)